In [6]:
# Cell 1
import os
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mn_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing import image
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt


In [7]:
# Cell 2
train_dir = "/content/drive/MyDrive/data/Training"   # contains subfolders: Aphids, Army worm, ...
val_dir = "/content/drive/MyDrive/data/Validation"   # same structure as training
classes = ["Aphids","Army worm","Bacterial blight","Cotton Boll Rot","Green Cotton Boll","Healthy","Powdery mildew","Target spot"]

IMG_SIZE = (224,224)   # 224x224 works for both MobileNetV2 and EfficientNetB0
BATCH_SIZE = 32
EPOCHS = 20
NUM_CLASSES = len(classes)

# where to save best models
save_dir = "./saved_models"
os.makedirs(save_dir, exist_ok=True)


In [8]:
# Cell 3
train_datagen = ImageDataGenerator(
    preprocessing_function=mn_preprocess,  # we'll adapt preprocessing per model later in model-specific code
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(preprocessing_function=mn_preprocess)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    classes=classes,
    class_mode='categorical',
    shuffle=True
)

validation_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    classes=classes,
    class_mode='categorical',
    shuffle=False
)


Found 6622 images belonging to 8 classes.
Found 277 images belonging to 8 classes.


In [9]:
# Cell 4
def build_model(base_model, input_shape=IMG_SIZE+(3,), num_classes=NUM_CLASSES, dropout=0.3):
    base_model.trainable = False  # freeze base
    inputs = keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = keras.Model(inputs, outputs)
    return model


In [10]:
# Cell 5
# Prepare MobileNetV2 base with appropriate preprocessing
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2 as MN
mn_base = MN(include_top=False, weights='imagenet', input_shape=IMG_SIZE+(3,))

model_mn = build_model(mn_base)

model_mn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

mn_ckpt = os.path.join(save_dir, "best_mobilenetv2.h5")
mn_callbacks = [
    ModelCheckpoint(mn_ckpt, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

# NOTE: train_datagen/val_datagen used earlier used mn_preprocess. If you change to EfficientNet, re-create generators with corresponding preprocess.
history_mn = model_mn.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    callbacks=mn_callbacks
)
# After training you can unfreeze some top layers and fine-tune (optional)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.3440 - loss: 1.8605
Epoch 1: val_accuracy improved from -inf to 0.77617, saving model to ./saved_models/best_mobilenetv2.h5


207/207 ━━━━━━━━━━━━━━━━━━━━ 1284s 6s/step - accuracy: 0.3447 - loss: 1.8587 - val_accuracy: 0.7762 - val_loss: 0.6730 - learning_rate: 1.0000e-04
Epoch 2/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7086 - loss: 0.8550
Epoch 2: val_accuracy improved from 0.77617 to 0.81588, saving model to ./saved_models/best_mobilenetv2.h5


207/207 ━━━━━━━━━━━━━━━━━━━━ 394s 2s/step - accuracy: 0.7087 - loss: 0.8547 - val_accuracy: 0.8159 - val_loss: 0.4833 - learning_rate: 1.0000e-04
Epoch 3/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8012 - loss: 0.5960
Epoch 3: val_accuracy improved from 0.81588 to 0.85560, saving model to ./saved_models/best_mobilenetv2.h5


207/207 ━━━━━━━━━━━━━━━━━━━━ 414s 2s/step - accuracy: 0.8012 - loss: 0.5960 - val_accuracy: 0.8556 - val_loss: 0.3662 - learning_rate: 1.0000e-04
Epoch 4/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8323 - loss: 0.4973
Epoch 4: val_accuracy improved from 0.85560 to 0.88087, saving model to ./saved_models/best_mobilenetv2.h5


...

207/207 ━━━━━━━━━━━━━━━━━━━━ 388s 2s/step - accuracy: 0.8960 - loss: 0.3048 - val_accuracy: 0.9134 - val_loss: 0.2419 - learning_rate: 1.0000e-04
Epoch 9/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8995 - loss: 0.2970
Epoch 9: val_accuracy improved from 0.91336 to 0.92419, saving model to ./saved_models/best_mobilenetv2.h5


207/207 ━━━━━━━━━━━━━━━━━━━━ 406s 2s/step - accuracy: 0.8995 - loss: 0.2970 - val_accuracy: 0.9242 - val_loss: 0.2210 - learning_rate: 1.0000e-04
Epoch 10/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9060 - loss: 0.2834
Epoch 10: val_accuracy did not improve from 0.92419
207/207 ━━━━━━━━━━━━━━━━━━━━ 397s 2s/step - accuracy: 0.9061 - loss: 0.2834 - val_accuracy: 0.9206 - val_loss: 0.2300 - learning_rate: 1.0000e-04
Epoch 11/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9132 - loss: 0.2527
Epoch 11: val_accuracy did not improve from 0.92419
207/207 ━━━━━━━━━━━━━━━━━━━━ 389s 2s/step - accuracy: 0.9132 - loss: 0.2527 - val_accuracy: 0.9206 - val_loss: 0.2369 - learning_rate: 1.0000e-04
Epoch 12/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9206 - loss: 0.2402
Epoch 12: val_accuracy improved from 0.92419 to 0.93141, saving model to ./saved_models/best_mobilenetv2.h5


207/207 ━━━━━━━━━━━━━━━━━━━━ 400s 2s/step - accuracy: 0.9206 - loss: 0.2403 - val_accuracy: 0.9314 - val_loss: 0.2182 - learning_rate: 1.0000e-04
Epoch 13/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9260 - loss: 0.2290
Epoch 13: val_accuracy did not improve from 0.93141
207/207 ━━━━━━━━━━━━━━━━━━━━ 458s 2s/step - accuracy: 0.9260 - loss: 0.2291 - val_accuracy: 0.9206 - val_loss: 0.2458 - learning_rate: 1.0000e-04
Epoch 14/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9167 - loss: 0.2376
Epoch 14: val_accuracy did not improve from 0.93141
207/207 ━━━━━━━━━━━━━━━━━━━━ 392s 2s/step - accuracy: 0.9167 - loss: 0.2375 - val_accuracy: 0.9314 - val_loss: 0.2135 - learning_rate: 1.0000e-04
Epoch 15/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9276 - loss: 0.2236
Epoch 15: val_accuracy improved from 0.93141 to 0.93502, saving model to ./saved_models/best_mobilenetv2.h5


207/207 ━━━━━━━━━━━━━━━━━━━━ 407s 2s/step - accuracy: 0.9276 - loss: 0.2236 - val_accuracy: 0.9350 - val_loss: 0.2088 - learning_rate: 1.0000e-04
Epoch 16/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9208 - loss: 0.2231
Epoch 16: val_accuracy did not improve from 0.93502
207/207 ━━━━━━━━━━━━━━━━━━━━ 407s 2s/step - accuracy: 0.9208 - loss: 0.2230 - val_accuracy: 0.9350 - val_loss: 0.1992 - learning_rate: 1.0000e-04
Epoch 17/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9354 - loss: 0.1956
Epoch 17: val_accuracy improved from 0.93502 to 0.94946, saving model to ./saved_models/best_mobilenetv2.h5


207/207 ━━━━━━━━━━━━━━━━━━━━ 402s 2s/step - accuracy: 0.9354 - loss: 0.1956 - val_accuracy: 0.9495 - val_loss: 0.1857 - learning_rate: 1.0000e-04
Epoch 18/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9331 - loss: 0.1999
Epoch 18: val_accuracy did not improve from 0.94946
207/207 ━━━━━━━━━━━━━━━━━━━━ 402s 2s/step - accuracy: 0.9332 - loss: 0.1999 - val_accuracy: 0.9458 - val_loss: 0.1809 - learning_rate: 1.0000e-04
Epoch 19/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9383 - loss: 0.1905
Epoch 19: val_accuracy did not improve from 0.94946
207/207 ━━━━━━━━━━━━━━━━━━━━ 402s 2s/step - accuracy: 0.9383 - loss: 0.1905 - val_accuracy: 0.9350 - val_loss: 0.2095 - learning_rate: 1.0000e-04
Epoch 20/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9397 - loss: 0.1783
Epoch 20: val_accuracy did not improve from 0.94946
207/207 ━━━━━━━━━━━━━━━━━━━━ 433s 2s/step - accuracy: 0.9397 - loss: 0.1783 - val_accuracy: 0.9386 - val_loss: 0.1939 - learning_rate: 1.0000e-04
Re

In [11]:
# Cell 6
# If you want EfficientNetB0, recreate datagens with its preprocess
train_datagen_eff = ImageDataGenerator(
    preprocessing_function=eff_preprocess,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)
val_datagen_eff = ImageDataGenerator(preprocessing_function=eff_preprocess)

train_generator_eff = train_datagen_eff.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    classes=classes,
    class_mode='categorical',
    shuffle=True
)

validation_generator_eff = val_datagen_eff.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    classes=classes,
    class_mode='categorical',
    shuffle=False
)

from tensorflow.keras.applications import EfficientNetB0 as ENB0
enb0_base = ENB0(include_top=False, weights='imagenet', input_shape=IMG_SIZE+(3,))

model_en = build_model(enb0_base)
model_en.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

en_ckpt = os.path.join(save_dir, "best_efficientnetb0.h5")
en_callbacks = [
    ModelCheckpoint(en_ckpt, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

history_en = model_en.fit(
    train_generator_eff,
    epochs=EPOCHS,
    validation_data=validation_generator_eff,
    callbacks=en_callbacks
)


Found 6622 images belonging to 8 classes.
Found 277 images belonging to 8 classes.
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.3818 - loss: 1.7691
Epoch 1: val_accuracy improved from -inf to 0.81949, saving model to ./saved_models/best_efficientnetb0.h5


207/207 ━━━━━━━━━━━━━━━━━━━━ 624s 3s/step - accuracy: 0.3827 - loss: 1.7672 - val_accuracy: 0.8195 - val_loss: 0.7348 - learning_rate: 1.0000e-04
Epoch 2/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7976 - loss: 0.7209
Epoch 2: val_accuracy improved from 0.81949 to 0.86282, saving model to ./saved_models/best_efficientnetb0.h5


207/207 ━━━━━━━━━━━━━━━━━━━━ 614s 3s/step - accuracy: 0.7977 - loss: 0.7206 - val_accuracy: 0.8628 - val_loss: 0.4239 - learning_rate: 1.0000e-04
Epoch 3/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8547 - loss: 0.4792
Epoch 3: val_accuracy improved from 0.86282 to 0.88809, saving model to ./saved_models/best_efficientnetb0.h5


207/207 ━━━━━━━━━━━━━━━━━━━━ 608s 3s/step - accuracy: 0.8547 - loss: 0.4791 - val_accuracy: 0.8881 - val_loss: 0.3283 - learning_rate: 1.0000e-04
Epoch 4/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8945 - loss: 0.3802
Epoch 4: val_accuracy did not improve from 0.88809
207/207 ━━━━━━━━━━━━━━━━━━━━ 600s 3s/step - accuracy: 0.8945 - loss: 0.3801 - val_accuracy: 0.8881 - val_loss: 0.2780 - learning_rate: 1.0000e-04
Epoch 5/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9083 - loss: 0.3095
Epoch 5: val_accuracy improved from 0.88809 to 0.89892, saving model to ./saved_models/best_efficientnetb0.h5


207/207 ━━━━━━━━━━━━━━━━━━━━ 618s 3s/step - accuracy: 0.9083 - loss: 0.3095 - val_accuracy: 0.8989 - val_loss: 0.2510 - learning_rate: 1.0000e-04
Epoch 6/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9100 - loss: 0.2848
Epoch 6: val_accuracy improved from 0.89892 to 0.90975, saving model to ./saved_models/best_efficientnetb0.h5


207/207 ━━━━━━━━━━━━━━━━━━━━ 606s 3s/step - accuracy: 0.9100 - loss: 0.2848 - val_accuracy: 0.9097 - val_loss: 0.2375 - learning_rate: 1.0000e-04
Epoch 7/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9250 - loss: 0.2382
Epoch 7: val_accuracy did not improve from 0.90975
207/207 ━━━━━━━━━━━━━━━━━━━━ 601s 3s/step - accuracy: 0.9250 - loss: 0.2382 - val_accuracy: 0.9061 - val_loss: 0.2320 - learning_rate: 1.0000e-04
Epoch 8/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9270 - loss: 0.2328
Epoch 8: val_accuracy improved from 0.90975 to 0.91336, saving model to ./saved_models/best_efficientnetb0.h5


207/207 ━━━━━━━━━━━━━━━━━━━━ 602s 3s/step - accuracy: 0.9270 - loss: 0.2328 - val_accuracy: 0.9134 - val_loss: 0.2227 - learning_rate: 1.0000e-04
Epoch 9/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9347 - loss: 0.2053
Epoch 9: val_accuracy improved from 0.91336 to 0.93502, saving model to ./saved_models/best_efficientnetb0.h5


207/207 ━━━━━━━━━━━━━━━━━━━━ 597s 3s/step - accuracy: 0.9347 - loss: 0.2053 - val_accuracy: 0.9350 - val_loss: 0.1932 - learning_rate: 1.0000e-04
Epoch 10/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9372 - loss: 0.1988
Epoch 10: val_accuracy did not improve from 0.93502
207/207 ━━━━━━━━━━━━━━━━━━━━ 629s 3s/step - accuracy: 0.9372 - loss: 0.1987 - val_accuracy: 0.9278 - val_loss: 0.1906 - learning_rate: 1.0000e-04
Epoch 11/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9454 - loss: 0.1763
Epoch 11: val_accuracy did not improve from 0.93502
207/207 ━━━━━━━━━━━━━━━━━━━━ 606s 3s/step - accuracy: 0.9454 - loss: 0.1763 - val_accuracy: 0.9278 - val_loss: 0.1961 - learning_rate: 1.0000e-04
Epoch 12/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9483 - loss: 0.1851
Epoch 12: val_accuracy did not improve from 0.93502
207/207 ━━━━━━━━━━━━━━━━━━━━ 600s 3s/step - accuracy: 0.9483 - loss: 0.1851 - val_accuracy: 0.9314 - val_loss: 0.1901 - learning_rate: 1.0000e-04
Ep

207/207 ━━━━━━━━━━━━━━━━━━━━ 589s 3s/step - accuracy: 0.9502 - loss: 0.1672 - val_accuracy: 0.9386 - val_loss: 0.1783 - learning_rate: 1.0000e-04
Epoch 14/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9534 - loss: 0.1506
Epoch 14: val_accuracy improved from 0.93863 to 0.94224, saving model to ./saved_models/best_efficientnetb0.h5


207/207 ━━━━━━━━━━━━━━━━━━━━ 628s 3s/step - accuracy: 0.9534 - loss: 0.1506 - val_accuracy: 0.9422 - val_loss: 0.1680 - learning_rate: 1.0000e-04
Epoch 15/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9615 - loss: 0.1350
Epoch 15: val_accuracy did not improve from 0.94224
207/207 ━━━━━━━━━━━━━━━━━━━━ 597s 3s/step - accuracy: 0.9615 - loss: 0.1351 - val_accuracy: 0.9422 - val_loss: 0.1856 - learning_rate: 1.0000e-04
Epoch 16/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9545 - loss: 0.1372
Epoch 16: val_accuracy did not improve from 0.94224
207/207 ━━━━━━━━━━━━━━━━━━━━ 597s 3s/step - accuracy: 0.9545 - loss: 0.1372 - val_accuracy: 0.9350 - val_loss: 0.1845 - learning_rate: 1.0000e-04
Epoch 17/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9486 - loss: 0.1499
Epoch 17: val_accuracy did not improve from 0.94224

Epoch 17: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.
207/207 ━━━━━━━━━━━━━━━━━━━━ 599s 3s/step - accuracy: 0.9486 - loss: 0

In [12]:
# Cell 7
# Load best weights (if training completed and files saved) and evaluate on validation set
from tensorflow.keras.models import load_model

mn_path = os.path.join(save_dir, "best_mobilenetv2.h5")
en_path = os.path.join(save_dir, "best_efficientnetb0.h5")

results = {}
if os.path.exists(mn_path):
    mn_loaded = load_model(mn_path)
    loss, acc = mn_loaded.evaluate(validation_generator, verbose=1)
    results['MobileNetV2'] = acc
else:
    print("MobileNetV2 model file not found at", mn_path)

if os.path.exists(en_path):
    en_loaded = load_model(en_path)
    loss, acc = en_loaded.evaluate(validation_generator_eff, verbose=1)
    results['EfficientNetB0'] = acc
else:
    print("EfficientNetB0 model file not found at", en_path)

print("Validation accuracies:", results)

# Choose best model
best_model_name = max(results, key=results.get) if results else None
best_model_path = mn_path if best_model_name == 'MobileNetV2' else en_path
print("Best model:", best_model_name, best_model_path)


9/9 ━━━━━━━━━━━━━━━━━━━━ 19s 2s/step - accuracy: 0.9163 - loss: 0.2837


9/9 ━━━━━━━━━━━━━━━━━━━━ 28s 2s/step - accuracy: 0.8697 - loss: 0.3429
Validation accuracies: {'MobileNetV2': 0.9494584798812866, 'EfficientNetB0': 0.9061371684074402}
Best model: MobileNetV2 ./saved_models/best_mobilenetv2.h5


In [14]:
# Cell 8 (fixed for Keras 3)
if best_model_path and os.path.exists(best_model_path):
    model = load_model(best_model_path)
    # Save in .keras format
    model.save(os.path.join(save_dir, "final_saved_model.h5"))
    print("Saved final model to final_saved_model.h5")
else:
    print("Best model not available to save.")

Saved final model to final_saved_model.keras


In [15]:
# Cell 9
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model
import tensorflow as tf

# config - point to your final model and the csv
MODEL_PATH = best_model_path  # or path to whichever model you want to use
CSV_PATH = "/content/drive/MyDrive/cotton_pesticides.csv"  # the CSV I created earlier

# Load model
model = load_model(MODEL_PATH)

# Load CSV
pest_df = pd.read_csv(CSV_PATH)
pest_df.set_index('class', inplace=True)

# helper for preprocessing depending on model
def preprocess_for_model(img_path, model_name='mobilenet'):
    img = image.load_img(img_path, target_size=IMG_SIZE)
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    if 'efficient' in model_name.lower():
        from tensorflow.keras.applications.efficientnet import preprocess_input as p
    else:
        from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as p
    x = p(x)
    return x

# prediction function
def predict_and_suggest(img_path, model, model_name='mobilenet'):
    x = preprocess_for_model(img_path, model_name=model_name)
    preds = model.predict(x)[0]  # softmax probs
    top_idx = np.argmax(preds)
    confidence = float(preds[top_idx])
    predicted_class = classes[top_idx]
    # Determine spray count using rule
    if predicted_class == "Healthy":
        sprays = 0
    else:
        if confidence >= 0.90:
            sprays = 1
        elif confidence >= 0.70:
            sprays = 2
        else:
            sprays = 3
    # Fetch pesticide row
    try:
        row = pest_df.loc[predicted_class]
        pesticide = row['pesticide']
        description = row['description']
        spray_rule = row['spray_rule']
    except KeyError:
        pesticide = "Unknown"
        description = ""
        spray_rule = ""
    result = {
        "predicted_class": predicted_class,
        "confidence": round(confidence*100,2),  # percent
        "recommended_pesticide": pesticide,
        "pesticide_description": description,
        "recommended_sprays": sprays,
        "spray_rule_text": spray_rule
    }
    return result

# Example usage:
# img_path = "/path/to/test_image.jpg"
# out = predict_and_suggest(img_path, model, model_name=best_model_name)
# print(out)


In [16]:
# Example usage:
img_path = "/content/drive/MyDrive/data/Validation/Army Worms/1.jpg"
out = predict_and_suggest(img_path, model, model_name=best_model_name)
print(out)

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
{'predicted_class': 'Army worm', 'confidence': 99.99, 'recommended_pesticide': 'Spinosad or Lambda-cyhalothrin', 'pesticide_description': 'Use insecticide effective against lepidopteran larvae. Follow safety instructions.', 'recommended_sprays': 1, 'spray_rule_text': '>=90:1 spray; 70-89:2 sprays; <70:3 sprays'}


In [17]:
# Cell 10 (optional fine-tune)
# Example fine-tune flow for MobileNetV2:
base_model = mn_base  # or enb0_base for EfficientNet
base_model.trainable = True

# Freeze up to a certain layer (fine-tune last N layers)
fine_tune_at = len(base_model.layers) - 30  # unfreeze last 30 layers
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model_mn.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-5),
                 loss='categorical_crossentropy',
                 metrics=['accuracy'])

fine_history = model_mn.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator,
    callbacks=[ModelCheckpoint(mn_ckpt, save_best_only=True, monitor='val_accuracy', mode='max')]
)


Epoch 1/10


KeyboardInterrupt: 